# Import Libraries and implement logger

In [6]:
## IMPORT LIBRARIES
import logging
from pathlib import Path
from io import StringIO
from datetime import datetime
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.utils.data import DataLoader, Dataset

In [7]:
# CREATE LOGGER FOR JUPYTER NOTEBOOKS
def setup_logger():
    # Create logger instance
    logger = logging.getLogger(__name__)
    logger.setLevel(logging.INFO)

    # Clear existing, avoid duplicate logs
    if logger.hasHandlers():
        logger.handlers.clear()
    
    # Create FileHandler in overrite mode
    file_handler = logging.FileHandler('logmodeler.log', mode ='w')

    # Set format
    formatter = logging.Formatter('%(asctime)s - %(levelno)s - %(lineno)d - %(module)s - %(message)s')
    file_handler.setFormatter(formatter)

    # Add FileHandler to logger
    logger.addHandler(file_handler)

    return logger

logger = setup_logger()

# Inspcet the data read in after the Visualzer

In [8]:
## Define function for inspection
def inspect_raw_data(df, output_path=None):
    '''
    Inspect a DataFrame and save info to a text file.
    Parameters:
    df : pandas DataFrame
    output_path : Path or str, optional
        Where to save the output file. If None and save_to_file is True,
        will use current directory
    '''

    # Set display options
    pd.set_option('display.max_rows', None)
    pd.set_option('display.max_columns', None)
    pd.set_option('display.width', None)
    
    # Prepare the information
    info_text = []
    info_text.append(f"Generated on: {datetime.now()}")
    info_text.append(f"Shape: {df.shape}")
    
    # Get info in string format
    buffer = StringIO()
    df.info(buf=buffer)
    info_text.append(f"\nDataFrame Info:")
    info_text.append(buffer.getvalue())

    # Add first 5 rows
    buffer_head = StringIO()
    df.head().to_string(buf=buffer_head)
    info_text.append(f"\nFirst 5 rows:")
    info_text.append(buffer_head.getvalue())

    # Add last 5 rows
    buffer_tail = StringIO()
    df.tail().to_string(buf=buffer_tail)
    info_text.append(f"\nLast 5 rows:")
    info_text.append(buffer_tail.getvalue())
    
    # Join all information
    full_report = '\n'.join(info_text)
    output_path = Path(output_path)
    output_path.write_text(full_report)
    logger.info(f"Report saved to: {output_path}")
    
    return full_report

In [9]:
# READ IN
data_path = Path('..') / '..' / 'data' / 'processed' / 'Visualizer' 
for file in data_path.glob('*.csv'):
    logger.info(f'Available csv file: {file.name}')

# ESO
# Normalized
X_train_eso_scaled = pd.read_csv(data_path / 'X_train_eso_scaled.csv', delimiter=',', parse_dates=['ID']) #, parse_dates=['ID']
X_test_eso_scaled = pd.read_csv(data_path / 'X_test_eso_scaled.csv', delimiter=',', parse_dates=['ID']) #, parse_dates=['ID']
# Not normalized
y_train_eso = pd.read_csv(data_path / 'y_train_eso.csv', delimiter=',', parse_dates=['ID']) #, parse_dates=['ID']
y_test_eso = pd.read_csv(data_path / 'y_test_eso.csv', delimiter=',', parse_dates=['ID']) #, parse_dates=['ID']

# GER
# Normalized
X_train_ger_scaled = pd.read_csv(data_path / 'X_train_ger_scaled.csv', delimiter=',', parse_dates=['ID']) #, parse_dates=['ID']
X_test_ger_scaled = pd.read_csv(data_path / 'X_test_ger_scaled.csv', delimiter=',', parse_dates=['ID']) #, parse_dates=['ID']
# Not normalized
y_train_ger = pd.read_csv(data_path / 'y_train_ger.csv', delimiter=',', parse_dates=['ID']) #, parse_dates=['ID']
y_test_ger = pd.read_csv(data_path / 'y_test_ger.csv', delimiter=',', parse_dates=['ID']) #, parse_dates=['ID']

In [10]:
# Save for inspection
# ESO
# Normalized
eso_scaled_Xtrain = 'X_train_eso_scaled'; eso_scaled_Xtest = 'X_test_eso_scaled'
_ = inspect_raw_data(X_train_eso_scaled, output_path=Path(f'{data_path}/../Modeler/{eso_scaled_Xtrain}.txt'))
_ = inspect_raw_data(X_test_eso_scaled, output_path=Path(f'{data_path}/../Modeler/{eso_scaled_Xtest}.txt'))
# GER
# Normalized
ger_scaled_Xtrain = 'X_train_ger_scaled'; ger_scaled_Xtest = 'X_test_ger_scaled'
_ = inspect_raw_data(X_train_ger_scaled, output_path=Path(f'{data_path}/../Modeler/{ger_scaled_Xtrain}.txt'))
_ = inspect_raw_data(X_test_ger_scaled, output_path=Path(f'{data_path}/../Modeler/{ger_scaled_Xtest}.txt'))

# Begin modeling after read in data
#### Requirements
    - Pipeline
    - Cross Validation
    - ML Regression
    - LSTM
    - Evaluation Metric
        - Basic of Neural Networks
        - !batch size, !epoche, !learning rate, !learning rate dk